# 05 — Nhãn tham chiếu
Tính nhãn tham chiếu tương ứng với từng cấu trúc, ghi rõ phương pháp, đơn vị và sai số.

**Phương pháp:** với mỗi vật liệu, lấy ma trận mật độ PBE đã hội tụ (bước 04) rồi đánh giá năng lượng của PBE, LDA và functional tham chiếu (mặc định r2SCAN) **trên cùng ma trận mật độ** (không SCF). Nhãn học là `Δ = E_ref[ρ_PBE] − E_PBE[ρ_PBE]`. Dùng cùng basis và cùng giả thế GTH-PBE cho cả ba nên hiệu năng lượng chỉ phản ánh khác biệt XC (cộng sai số xem bên dưới).

**Hạn chế cần biết:** (1) giả thế GTH-PBE không được xây cho r2SCAN; (2) đánh giá không tự hợp bỏ qua việc ρ thay đổi khi đổi functional; (3) r2SCAN còn phụ thuộc τ, mô hình ở bước 07 chỉ thấy ρ và ∇ρ.

**Đầu ra:** `data/reference_labels/labels.csv`, `scf_check.csv`, `labels_method.json`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q pyscf pymatgen

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd, time
from utils import dft_utils as dft, density_utils as du, structure_utils as su

DENS, LAB = f"{DATA}/densities", f"{DATA}/reference_labels"
REF_XC = "r2scan"           # functional tham chiếu (PySCF/libxc)
BASELINE_XC = "lda,vwn"     # baseline thứ hai để so sánh ở bước 08
SCF_CHECK_N = 3             # số mẫu nhỏ nhất chạy SCF đầy đủ với REF_XC để ước lượng sai số không tự hợp
HA2EV = dft.HA2EV

settings = json.load(open(f"{PROC}/pbe_settings.json"))["settings"]
sel = pd.read_csv(f"{PROC}/selected.csv")
ids = [m for m in sel.material_id
       if os.path.exists(f"{DENS}/{m}.json") and json.load(open(f"{DENS}/{m}.json"))["converged"]]
print("Mẫu có mật độ PBE hội tụ:", len(ids), "/", len(sel))

## 1. Năng lượng PBE / LDA / tham chiếu trên cùng mật độ

In [ ]:
os.makedirs(LAB, exist_ok=True)
rows = []
for k, mid in enumerate(ids):
    d = du.load_density(f"{DENS}/{mid}.npz")
    info = json.load(open(f"{DENS}/{mid}.json"))
    struct, cell = dft.cell_from_arrays(d["lattice_A"], d["species"], d["frac_coords"], settings)
    spin = bool(settings["spin"])
    t0 = time.time()
    e = {xc: dft.energy_on_density(cell, d["kpts"], d["dm_kpts"], xc, spin) for xc in ("pbe", BASELINE_XC, REF_XC)}
    n = len(struct)
    rows.append({"material_id": mid, "n_atoms": n,
                 "e_pbe_Ha": e["pbe"], "e_lda_Ha": e[BASELINE_XC], "e_ref_Ha": e[REF_XC],
                 "delta_ref_pbe_Ha": e[REF_XC] - e["pbe"], "delta_lda_pbe_Ha": e[BASELINE_XC] - e["pbe"],
                 "delta_ref_pbe_eV_per_atom": (e[REF_XC] - e["pbe"]) * HA2EV / n,
                 "delta_lda_pbe_eV_per_atom": (e[BASELINE_XC] - e["pbe"]) * HA2EV / n,
                 "check_pbe_vs_scf_Ha": e["pbe"] - info["e_tot_Ha"]})
    if k % 20 == 0 or k == len(ids) - 1:
        print(f"[{k+1}/{len(ids)}] {mid}  Δ_ref-PBE = {rows[-1]['delta_ref_pbe_eV_per_atom']*1e3:8.2f} meV/atom  ({time.time()-t0:.1f}s)")
labels = pd.DataFrame(rows)
bad = labels[labels.check_pbe_vs_scf_Ha.abs() > 1e-6]
print("\nKiểm tra E_PBE tính lại so với E_PBE của SCF: lệch lớn nhất =", labels.check_pbe_vs_scf_Ha.abs().max(), "Ha")
if len(bad):
    print("CẢNH BÁO: các mẫu sau lệch > 1e-6 Ha (có thể do smearing/entropy), kiểm tra trước khi dùng:", list(bad.material_id))
labels.to_csv(f"{LAB}/labels.csv", index=False)

## 2. Ước lượng sai số của đánh giá không tự hợp
Chạy SCF đầy đủ với `REF_XC` cho vài mẫu nhỏ nhất và so với giá trị không tự hợp. Số mẫu ít nên đây chỉ là chỉ báo thô cho mức sai số, không phải thống kê đầy đủ.

In [ ]:
chk_rows = []
small = sel[sel.material_id.isin(ids)].sort_values(["n_atoms", "n_elements"]).head(SCF_CHECK_N)
for r in small.itertuples():
    struct = su.read_structure(f"{RAW}/cif/{r.material_id}.cif")
    info, *_ = dft.run_scf(struct, r.band_gap_eV, {**settings, "xc": REF_XC})
    e_nsc = float(labels.loc[labels.material_id == r.material_id, "e_ref_Ha"].iloc[0])
    chk_rows.append({"material_id": r.material_id, "e_ref_scf_Ha": info["e_tot_Ha"], "e_ref_nsc_Ha": e_nsc,
                     "diff_meV_per_atom": (e_nsc - info["e_tot_Ha"]) * HA2EV * 1e3 / len(struct),
                     "scf_converged": info["converged"]})
    print(chk_rows[-1])
chk = pd.DataFrame(chk_rows)
chk.to_csv(f"{LAB}/scf_check.csv", index=False)
scf_mae = float(chk.diff_meV_per_atom.abs().mean()) if len(chk) else None
print("MAE |E_nsc - E_scf| =", scf_mae, "meV/nguyên tử trên", len(chk), "mẫu")

In [ ]:
method = {
    "label": "delta_ref_pbe = E_ref[rho_PBE] - E_PBE[rho_PBE]",
    "reference_xc": REF_XC, "baseline_xc": BASELINE_XC, "density": "PBE tự hợp (bước 04), cùng ma trận mật độ cho mọi functional",
    "basis": settings["basis"], "pseudo": settings["pseudo"],
    "units": {"energy": "Ha (cột *_eV_per_atom đã đổi sang eV/nguyên tử, 1 Ha = 27.211386 eV)"},
    "uncertainty": {"scf_check_n": len(chk), "scf_check_mae_meV_per_atom": scf_mae,
                    "meaning": "sai số do không tự hợp (chỉ báo thô, ít mẫu)"},
    "caveats": ["GTH-PBE pseudopotential dùng với functional tham chiếu",
                "functional tham chiếu phụ thuộc tau trong khi mô hình ML chỉ dùng rho và |grad rho|",
                "năng lượng không gồm đóng góp entropy của smearing"],
    "n_labels": int(len(labels)), "date": time.strftime("%Y-%m-%d"),
}
json.dump(method, open(f"{LAB}/labels_method.json", "w"), indent=2, ensure_ascii=False)
labels[["delta_ref_pbe_eV_per_atom", "delta_lda_pbe_eV_per_atom"]].describe().round(4)